In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam, SGD, RMSprop
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.multioutput import MultiOutputRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

%matplotlib inline

2025-09-16 18:15:40.307493: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758046540.334000   12010 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758046540.341895   12010 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
df1=pd.read_excel("/kaggle/input/rainfall-prediction-challenge-2025/Data Aktual.xlsx")
df2=pd.read_excel("/kaggle/input/rainfall-prediction-challenge-2025/Data Input Hybrid.xlsx")

In [3]:
df1.head()

,Date,Aktual_Y1,Aktual_Y2,Aktual_Y3,Aktual_Y4
0,2013-01-01,57.83,62.25,66.28,66.28
1,2013-01-02,47.53,53.67,57.54,57.54
2,2013-01-03,5.33,7.20,7.92,7.92
3,2013-01-04,12.54,8.85,6.92,6.92
4,2013-01-05,7.43,6.84,8.08,8.08


In [4]:
df1.shape

(4017, 5)

In [5]:
df1.isnull().sum()

Date         0
Aktual_Y1    0
Aktual_Y2    0
Aktual_Y3    0
Aktual_Y4    0
dtype: int64

In [6]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4017 entries, 0 to 4016
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       4017 non-null   datetime64[ns]
 1   Aktual_Y1  4017 non-null   float64       
 2   Aktual_Y2  4017 non-null   float64       
 3   Aktual_Y3  4017 non-null   float64       
 4   Aktual_Y4  4017 non-null   float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 157.0 KB


In [7]:
df2.head()

,Tanggal,w1,w2,w3,w4,ehat1,eresid1,ehat2,eresid2,ehat3,eresid3,ehat4,eresid4
0,2013-01-05,-0.536191,-5.487963,-3.669789,-4.130197,2.018349,-26.844071,-0.516930,-4.264845,-0.229540,-2.374609,-0.022629,-1.343081
1,2013-01-06,-3.327927,-7.747863,-7.029111,-7.321747,2.200193,-5.469971,-0.246884,8.296666,-0.367963,3.662227,-0.040259,3.613250
2,2013-01-07,5.528366,1.756227,4.811639,3.619966,0.934243,-20.512956,-0.703296,-5.540482,0.149313,-4.300833,0.026537,-3.561647
3,2013-01-08,-4.972129,-7.771838,-7.604647,-7.611148,1.481614,-22.569946,-0.089907,-3.900944,-0.307566,-4.152582,-0.046692,-3.072028
4,2013-01-09,-3.951102,-8.223785,-6.535069,-7.321102,1.747543,-20.661473,-0.380272,-1.043197,-0.203747,-2.759564,-0.044268,-2.269412


In [8]:
df2.shape

(4013, 13)

In [9]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4013 entries, 0 to 4012
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Tanggal  4013 non-null   datetime64[ns]
 1   w1       4013 non-null   float64       
 2   w2       4013 non-null   float64       
 3   w3       4013 non-null   float64       
 4   w4       4013 non-null   float64       
 5   ehat1    4013 non-null   float64       
 6   eresid1  4013 non-null   float64       
 7   ehat2    4013 non-null   float64       
 8   eresid2  4013 non-null   float64       
 9   ehat3    4013 non-null   float64       
 10  eresid3  4013 non-null   float64       
 11  ehat4    4013 non-null   float64       
 12  eresid4  4013 non-null   float64       
dtypes: datetime64[ns](1), float64(12)
memory usage: 407.7 KB


In [10]:
df2.head()

,Tanggal,w1,w2,w3,w4,ehat1,eresid1,ehat2,eresid2,ehat3,eresid3,ehat4,eresid4
0,2013-01-05,-0.536191,-5.487963,-3.669789,-4.130197,2.018349,-26.844071,-0.516930,-4.264845,-0.229540,-2.374609,-0.022629,-1.343081
1,2013-01-06,-3.327927,-7.747863,-7.029111,-7.321747,2.200193,-5.469971,-0.246884,8.296666,-0.367963,3.662227,-0.040259,3.613250
2,2013-01-07,5.528366,1.756227,4.811639,3.619966,0.934243,-20.512956,-0.703296,-5.540482,0.149313,-4.300833,0.026537,-3.561647
3,2013-01-08,-4.972129,-7.771838,-7.604647,-7.611148,1.481614,-22.569946,-0.089907,-3.900944,-0.307566,-4.152582,-0.046692,-3.072028
4,2013-01-09,-3.951102,-8.223785,-6.535069,-7.321102,1.747543,-20.661473,-0.380272,-1.043197,-0.203747,-2.759564,-0.044268,-2.269412


In [11]:
df2['Tanggal'] = pd.to_datetime(df2['Tanggal'], format='%d-%b-%Y')
df2['Day'] = df2['Tanggal'].dt.day
df2['Month'] = df2['Tanggal'].dt.month
df2['Year'] = df2['Tanggal'].dt.year

In [12]:
df2.drop(columns=["Tanggal"],axis=1,inplace=True)

In [13]:
df2.head()

,w1,w2,w3,w4,ehat1,eresid1,ehat2,eresid2,ehat3,eresid3,ehat4,eresid4,Day,Month,Year
0,-0.536191,-5.487963,-3.669789,-4.130197,2.018349,-26.844071,-0.516930,-4.264845,-0.229540,-2.374609,-0.022629,-1.343081,5,1,2013
1,-3.327927,-7.747863,-7.029111,-7.321747,2.200193,-5.469971,-0.246884,8.296666,-0.367963,3.662227,-0.040259,3.613250,6,1,2013
2,5.528366,1.756227,4.811639,3.619966,0.934243,-20.512956,-0.703296,-5.540482,0.149313,-4.300833,0.026537,-3.561647,7,1,2013
3,-4.972129,-7.771838,-7.604647,-7.611148,1.481614,-22.569946,-0.089907,-3.900944,-0.307566,-4.152582,-0.046692,-3.072028,8,1,2013
4,-3.951102,-8.223785,-6.535069,-7.321102,1.747543,-20.661473,-0.380272,-1.043197,-0.203747,-2.759564,-0.044268,-2.269412,9,1,2013


In [14]:
input_cols = ['w1', 'w2', 'w3', 'w4', 'ehat1', 'ehat2', 'ehat3', 'ehat4']
target_cols = ['eresid1', 'eresid2', 'eresid3', 'eresid4']

scaler=MinMaxScaler()
df2[input_cols]=scaler.fit_transform(df2[input_cols])

In [15]:
df2.head()

,w1,w2,w3,w4,ehat1,eresid1,ehat2,eresid2,ehat3,eresid3,ehat4,eresid4,Day,Month,Year
0,0.155260,0.111268,0.140664,0.127793,0.837998,-26.844071,0.589346,-4.264845,0.225970,-2.374609,0.129884,-1.343081,5,1,2013
1,0.132941,0.095580,0.118107,0.105804,0.850875,-5.469971,0.668414,8.296666,0.193665,3.662227,0.112061,3.613250,6,1,2013
2,0.203744,0.161554,0.197615,0.181188,0.761234,-20.512956,0.534779,-5.540482,0.314384,-4.300833,0.179590,-3.561647,7,1,2013
3,0.119797,0.095414,0.114242,0.103811,0.799993,-22.569946,0.714376,-3.900944,0.207760,-4.152582,0.105557,-3.072028,8,1,2013
4,0.127959,0.092277,0.121424,0.105809,0.818823,-20.661473,0.629359,-1.043197,0.231989,-2.759564,0.108008,-2.269412,9,1,2013


In [16]:
X = df2[input_cols]
y = df2[target_cols]

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True,random_state=42)

In [18]:
base_model = CatBoostRegressor(iterations=1000,learning_rate=0.005,depth=6,verbose=500)

multi_model = MultiOutputRegressor(base_model)  

multi_model.fit(X_train, y_train)

0:	learn: 8.5513541	total: 48.4ms	remaining: 48.4s
500:	learn: 7.9076158	total: 800ms	remaining: 797ms
999:	learn: 7.5770018	total: 1.63s	remaining: 0us
0:	learn: 6.7946891	total: 1.76ms	remaining: 1.76s
500:	learn: 6.3985141	total: 758ms	remaining: 755ms
999:	learn: 6.1986917	total: 1.51s	remaining: 0us
0:	learn: 6.5448734	total: 2.27ms	remaining: 2.27s
500:	learn: 6.1845656	total: 759ms	remaining: 756ms
999:	learn: 5.9997718	total: 1.51s	remaining: 0us
0:	learn: 6.4714262	total: 2.28ms	remaining: 2.27s
500:	learn: 6.1055870	total: 755ms	remaining: 752ms
999:	learn: 5.9222452	total: 1.5s	remaining: 0us


MultiOutputRegressor(estimator=<catboost.core.CatBoostRegressor object at 0x7b76fb98a710>)

In [19]:
y_true_all = y_test[target_cols].values.flatten()
y_pred = multi_model.predict(X_test)

y_pred_all = y_pred.flatten()

rmse_all = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
mae_all = mean_absolute_error(y_true_all, y_pred_all)
r2_all = r2_score(y_true_all, y_pred_all)
mape_all = (np.abs(y_true_all - y_pred_all) / np.maximum(y_true_all, 1e-6)).mean() * 100


print(f"RMSE : {rmse_all:.4f}")
print(f"MAE  : {mae_all:.4f}")
print(f"R2   : {r2_all:.4f}")
print(f"MAPE : {mape_all:.2f}%")

RMSE : 6.5999
MAE  : 3.9334
R2   : 0.0417
MAPE : 188714313.08%


In [20]:
# Convert DataFrame to NumPy array before reshaping
X_train_np = X_train.values
X_test_np  = X_test.values

# Reshape for LSTM: (samples, timesteps, features)
# Here timesteps = 1
X_train_3d = X_train_np.reshape((X_train_np.shape[0], 1, X_train_np.shape[1]))
X_test_3d  = X_test_np.reshape((X_test_np.shape[0], 1, X_test_np.shape[1]))

print(f"X_train shape: {X_train_3d.shape}")
print(f"X_test shape:  {X_test_3d.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")


X_train shape: (3210, 1, 8)
X_test shape:  (803, 1, 8)
y_train shape: (3210, 4)
y_test shape:  (803, 4)


In [21]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, BatchNormalization
from keras.optimizers import Adam

n_features = X_train_3d.shape[2]
n_outputs  = y_train.shape[1]

model = Sequential([
    LSTM(128, activation='tanh', return_sequences=True, input_shape=(X_train_3d.shape[1], n_features)),
    Dropout(0.2),
    BatchNormalization(),
    
    LSTM(64, activation='tanh'),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dense(n_outputs)  
])


optimizer = Adam(learning_rate=5e-5)
model.compile(optimizer=optimizer,loss='mean_squared_error',metrics=['mean_squared_error'])

model.summary()


2025-09-16 18:15:53.189776: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 1, 128)         │        70,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,276 (477.64 KB)

 Trainable params: 122,020 (476.64 KB)

 Non-trainable params: 256 (1.00 KB)

In [22]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

reduce_lr = ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=10,min_lr=1e-6,verbose=1)

In [23]:
history = model.fit(X_train_3d, y_train,epochs=150,batch_size=64,validation_data=(X_test_3d, y_test),verbose=2, callbacks=[early_stop, reduce_lr])

Epoch 1/150
51/51 - 7s - 129ms/step - loss: 51.0325 - mean_squared_error: 51.0325 - val_loss: 45.4561 - val_mean_squared_error: 45.4561 - learning_rate: 5.0000e-05
Epoch 2/150
51/51 - 0s - 6ms/step - loss: 51.0334 - mean_squared_error: 51.0334 - val_loss: 45.4559 - val_mean_squared_error: 45.4559 - learning_rate: 5.0000e-05
Epoch 3/150
51/51 - 0s - 6ms/step - loss: 51.0353 - mean_squared_error: 51.0353 - val_loss: 45.4558 - val_mean_squared_error: 45.4558 - learning_rate: 5.0000e-05
Epoch 4/150
51/51 - 0s - 6ms/step - loss: 51.0248 - mean_squared_error: 51.0248 - val_loss: 45.4554 - val_mean_squared_error: 45.4554 - learning_rate: 5.0000e-05
Epoch 5/150
51/51 - 0s - 6ms/step - loss: 51.0305 - mean_squared_error: 51.0305 - val_loss: 45.4551 - val_mean_squared_error: 45.4551 - learning_rate: 5.0000e-05
Epoch 6/150
51/51 - 0s - 6ms/step - loss: 51.0338 - mean_squared_error: 51.0338 - val_loss: 45.4542 - val_mean_squared_error: 45.4542 - learning_rate: 5.0000e-05
Epoch 7/150
51/51 - 0s - 6

In [24]:
y_pred = model.predict(X_test_3d)
rmse_overall = np.sqrt(mean_squared_error(y_test, y_pred))
mae_overall  = mean_absolute_error(y_test, y_pred)
r2_overall   = r2_score(y_test, y_pred)
mape_overall = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("Overall Metrics:")
print(f"RMSE : {rmse_overall:.4f}")
print(f"MAE  : {mae_overall:.4f}")
print(f"R2   : {r2_overall:.4f}")
print(f"MAPE : {mape_overall:.2f}%")

26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step 
Overall Metrics:
RMSE : 6.6161
MAE  : 3.9160
R2   : 0.0357
MAPE : 297.07%


In [41]:
X_submission = df2[input_cols].iloc[-100:].values
X_submission_3d = X_submission.reshape((X_submission.shape[0], 1, X_submission.shape[1]))

y_pred_submission = model.predict(X_submission_3d)

submission_df = pd.DataFrame({
    'id': range(len(y_pred_submission)),
    'Y1': y_pred_submission[:, 0],
    'Y2': y_pred_submission[:, 1],
    'Y3': y_pred_submission[:, 2],
    'Y4': y_pred_submission[:, 3]
})

submission_df.to_csv("lstm_submission.csv", index=False)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


In [42]:
df=pd.read_csv("/kaggle/working/lstm_submission.csv")

In [43]:
df.head()

,id,Y1,Y2,Y3,Y4
0,0,1.112891,1.197157,1.063259,1.058061
1,1,0.709734,0.658697,0.626224,0.689312
2,2,0.909269,0.928712,0.849294,0.868365
3,3,0.952969,0.976678,0.884238,0.903448
4,4,1.122937,1.206078,1.067503,1.065986
